# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayushmanlohani/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one page/content item for a given month. I will use a mid-panel month, such as March 2026, to study the page's search performance and freshness.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, avg_position, ctr, content_age_days, days_since_last_update.
Label: is_declining_label, derived from trend_direction.
Context: content_id and month, used to identify the page and time period.
Excluded: client_id, because it identifies the client rather than describing the page, and trend_direction as a model input because it directly defines the label and would cause leakage.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# Query 1 — Grain
print("QUERY 1 — Grain check")
print(
    con.sql(f"""
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS row_count
        FROM {MAR}
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
        LIMIT 5
    """).df()
)

# Query 2 — March row count + date span
print("\nQUERY 2 — March count and date span")
print(
    con.sql(f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(DISTINCT content_hash_id) AS content_items,
            MIN(report_date) AS first_date,
            MAX(report_date) AS last_date
        FROM {MAR}
    """).df()
)

# Query 3 — Availability
print("\nQUERY 3 — Available search data")
print(
    con.sql(f"""
        SELECT
            COUNT(*) AS available_rows,
            COUNT(DISTINCT content_hash_id) AS available_content_items
        FROM {MAR}
        WHERE gsc_data_available IS TRUE
    """).df()
)

QUERY 1 — Grain check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []

QUERY 2 — March count and date span
   row_count  content_items first_date  last_date
0    9841378         331437 2026-03-01 2026-03-31

QUERY 3 — Available search data
   available_rows  available_content_items
0         3611061                   176738


In [8]:
con.sql(f"DESCRIBE SELECT * FROM {MAR}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data may not capture every reason a page's search performance changes. Some fields have missing or uneven availability, and the available time window may not represent longer-term seasonal patterns.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.